In [78]:
pip install openmeteo-requests requests-cache retry-requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [79]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.73061,
	"longitude": -73.935242,
	"start_date": "2020-01-01",
	"end_date": "2025-12-31",
	"daily": [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",   
    "sunrise",
    "sunset"
],
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "wind_speed_10m"],
	"timezone": "auto",
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_rain = hourly.Variables(5).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(6).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(8).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()

daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()   # ✅ ADD THIS
daily_sunrise = daily.Variables(4).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(5).ValuesInt64AsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation"] = daily_precipitation_sum
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_dataframe = pd.DataFrame(data = daily_data)

hourly_dataframe.head()

Coordinates: 40.738136291503906°N -73.91488647460938°E
Elevation: 14.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s


,date,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,wind_speed_10m
0,2020-01-01 00:00:00+00:00,39.829998,79.699532,34.070000,32.075100,0.0,0.0,0.0,0.0,10.088845
1,2020-01-01 01:00:00+00:00,38.209999,75.076363,31.010000,29.407892,0.0,0.0,0.0,0.0,11.428421
2,2020-01-01 02:00:00+00:00,35.779999,77.071312,29.299999,26.927099,0.0,0.0,0.0,0.0,10.963583
3,2020-01-01 03:00:00+00:00,34.790001,78.990501,28.940001,26.067801,0.0,0.0,0.0,0.0,10.535296
4,2020-01-01 04:00:00+00:00,34.160000,77.779877,27.950001,24.650108,0.0,0.0,0.0,0.0,12.081872


In [80]:
import requests
import pandas as pd

url = "https://data.cityofnewyork.us/resource/7ym2-wayt.json?$limit=50000"

response = requests.get(url)
data = response.json()

df_traffic = pd.DataFrame(data)

print(df_traffic.head())
print(df_traffic.columns)

  requestid    boro    yr  m  d hh  mm vol segmentid  \
0     12512  Queens  2013  3  7  4  15   5     55135   
1     12512  Queens  2013  3  7  4  30   8     55135   
2     12512  Queens  2013  3  7  4  45   8     55135   
3     12512  Queens  2013  3  7  5   0   7     55135   
4     12512  Queens  2013  3  7  5  15   9     55135   

                      wktgeom  street     fromst           tost direction  
0  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
1  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
2  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
3  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
4  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
Index(['requestid', 'boro', 'yr', 'm', 'd', 'hh', 'mm', 'vol', 'segmentid',
       'wktgeom', 'street', 'fromst', 'tost', 'direction'],
      dtype='object')


In [81]:
df_traffic["datetime"] = pd.to_datetime(
    df_traffic["yr"].astype(str) + "-" +
    df_traffic["m"].astype(str) + "-" +
    df_traffic["d"].astype(str) + " " +
    df_traffic["hh"].astype(str) + ":" +
    df_traffic["mm"].astype(str)
)

In [82]:
df_traffic["datetime"] = df_traffic["datetime"].dt.tz_localize(None)
hourly_dataframe["date"] = hourly_dataframe["date"].dt.tz_localize(None)

In [83]:
df_traffic["vol"] = pd.to_numeric(df_traffic["vol"], errors="coerce")

df_traffic = df_traffic.dropna(subset=["datetime", "vol"])

In [84]:
df_merged = pd.merge_asof(
    df_traffic.sort_values("datetime"),
    hourly_dataframe.sort_values("date"),
    left_on="datetime",
    right_on="date",
    direction="nearest"
)

In [85]:
print(df_merged.head())

  requestid       boro    yr  m  d hh  mm  vol segmentid  \
0      2276  Manhattan  2000  1  1  0  15  231     32956   
1      2276  Manhattan  2000  1  1  0  30  238     32956   
2      2276  Manhattan  2000  1  1  0  45  249     32956   
3      2276  Manhattan  2000  1  1  1   0  231     32956   
4      2276  Manhattan  2000  1  1  1  15  196     32956   

                     wktgeom  ...       date temperature_2m  \
0  POINT (986884.7 207042.2)  ... 2020-01-01      39.829998   
1  POINT (986884.7 207042.2)  ... 2020-01-01      39.829998   
2  POINT (986884.7 207042.2)  ... 2020-01-01      39.829998   
3  POINT (986884.7 207042.2)  ... 2020-01-01      39.829998   
4  POINT (986884.7 207042.2)  ... 2020-01-01      39.829998   

  relative_humidity_2m dew_point_2m apparent_temperature precipitation  rain  \
0            79.699532        34.07              32.0751           0.0   0.0   
1            79.699532        34.07              32.0751           0.0   0.0   
2            79.6995

In [86]:
pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [88]:
df_traffic["date"] = df_traffic["datetime"].dt.date

df_traffic_daily = df_traffic.groupby("date").agg({
    "vol": "sum"
}).reset_index()

df_traffic_daily["date"] = pd.to_datetime(df_traffic_daily["date"])

df_traffic_daily.head()

,date,vol
0,2000-01-01,19159
1,2000-01-06,13320
2,2000-01-07,6975
3,2006-09-15,10785
4,2006-09-16,19226


In [89]:
df_weather_daily = daily_dataframe[[
    "date",
    "temperature_2m_mean",
    "precipitation"
]].copy()

df_weather_daily = df_weather_daily.dropna()
df_weather_daily.head()

,date,temperature_2m_mean,precipitation
0,2020-01-01 00:00:00+00:00,36.076248,0.000000
1,2020-01-02 00:00:00+00:00,36.361252,0.000000
2,2020-01-03 00:00:00+00:00,43.625000,0.177165
3,2020-01-04 00:00:00+00:00,46.459999,0.244094
4,2020-01-05 00:00:00+00:00,36.916248,0.043307


In [90]:
df_traffic_daily["date"] = pd.to_datetime(df_traffic_daily["date"])
df_traffic_daily = df_traffic_daily.dropna()
df_traffic_daily.head()

,date,vol
0,2000-01-01,19159
1,2000-01-06,13320
2,2000-01-07,6975
3,2006-09-15,10785
4,2006-09-16,19226


In [91]:
df_traffic_daily["date"] = df_traffic_daily["date"].dt.tz_localize(None)
df_weather_daily["date"] = df_weather_daily["date"].dt.tz_localize(None)

In [92]:
df_final = pd.merge(
    df_traffic_daily,
    df_weather_daily,
    on="date",
    how="inner"
)

df_final.head()

,date,vol,temperature_2m_mean,precipitation
0,2020-01-26,10329,39.136250,0.000000
1,2020-01-27,15411,37.793751,0.000000
2,2020-01-28,2337,37.212502,0.000000
3,2020-01-31,13097,36.147499,0.043307
4,2020-08-17,4045,69.901245,0.248032


In [95]:
df_final = df_final[df_final["vol"] > 0]
df_final = df_final.dropna()

df_final.describe()

,date,vol,temperature_2m_mean,precipitation,predicted_vol
count,161,161.000000,161.000000,161.000000,161.000000
mean,2022-07-23 04:46:12.670807552,6156.416149,56.812283,0.120825,6156.416504
min,2020-01-26 00:00:00,11.000000,19.429998,0.000000,1600.824951
25%,2021-10-01 00:00:00,525.000000,45.361248,0.000000,4773.909180
50%,2022-05-21 00:00:00,2999.000000,58.302498,0.003937,6350.537109
75%,2023-06-19 00:00:00,7186.000000,69.244995,0.094488,7708.113770
max,2024-06-07 00:00:00,98083.000000,81.920006,1.488189,9283.869141
std,NaN,11126.907522,14.486526,0.254161,1788.256348


In [93]:
from sklearn.linear_model import LinearRegression

X = df_final[["precipitation", "temperature_2m_mean"]]
y = df_final["vol"]

model = LinearRegression()
model.fit(X, y)

print("Coefficients:", model.coef_)

Coefficients: [-341.85178   122.969894]


In [94]:
df_final["predicted_vol"] = model.predict(X)

In [96]:
df_final.to_csv("weather_traffic_daily.csv", index=False)